In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import re

# Ahora importa el módulo
from RRAM.Representate import config_ax

def setup_paper_plt(plt, latex=True, scaling=1):
    plt.rcParams.update(
        {
            "pgf.texsystem": "pdflatex",
            "text.usetex": latex,
            "font.family": "mathpazo",
            "text.latex.preamble": "\n".join(
                [
                    r"\usepackage[utf8]{inputenc}",
                    r"\usepackage[T1]{fontenc}",
                    r"\usepackage{siunitx}",
                    r"\usepackage{physics}",
                ]
            ),
        }
    )

    SMALL_SIZE = 8 * scaling
    MEDIUM_SIZE = 10 * scaling
    BIGGER_SIZE = 11 * scaling
    BIGGEST_SIZE = 14 * scaling
    
    plt.rc("font", size=SMALL_SIZE)
    plt.rc("axes", titlesize=MEDIUM_SIZE)
    plt.rc("axes", labelsize=BIGGEST_SIZE*1.2)
    plt.rc("xtick", labelsize=BIGGEST_SIZE)
    plt.rc("ytick", labelsize=BIGGEST_SIZE)
    plt.rc("legend", fontsize=SMALL_SIZE)
    plt.rc("figure", titlesize=BIGGER_SIZE)
    plt.rc("axes", titlesize=BIGGER_SIZE)
    
setup_paper_plt(plt, latex=True, scaling=2)

In [2]:
# Rutas de los archivos
ruta_datos = Path.cwd() / "Datos_Experimentales" / "Medidas_Experimentales_RRAM"
ruta_figuras = Path.cwd() / "Datos_Experimentales" / "V_Reset"

if ruta_figuras.exists():
    shutil.rmtree(ruta_figuras)
    ruta_figuras.mkdir(exist_ok=True)
else:
    ruta_figuras.mkdir(exist_ok=True)

# Listar todos los archivos
archivos = []
for archivo in ruta_datos.glob("Cycle_n_*.txt"):
    # Buscar el número dentro del nombre (después de "Cycle_p_")
    match = re.fullmatch(r"Cycle_n_(\d{3,4})\.txt", archivo.name)
    if match:
        numero = int(match.group(1))  # convertir a entero
        if 900 <= numero <= 954:  # aplicar filtro de rango
            archivos.append(archivo)

print(f"Archivos encontrados: {len(archivos)}")
print(archivos)


Archivos encontrados: 55
[WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_n_900.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_n_901.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_n_902.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_n_903.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_n_904.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_n_905.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_n_906.txt'), WindowsPath('c:/Users/Usuario/Documents/GitHub/RRAM_Simul

In [3]:
def plot_iv_and_derivative(
    voltaje, intensidad, dI_dV, top_indices, ruta_figuras, **kwargs
):
    """
    Plots the I-V curve and its derivative, highlighting specific points with the highest derivative values.
    Parameters:
    -----------
    V_set : array-like
        Array of voltage values (x-axis data).
    I_set : array-like
        Array of current values corresponding to the voltage values (y-axis data for the I-V curve).
    dI_dV : array-like
        Array of derivative values (y-axis data for the derivative plot).
    top_indices : list of int
        Indices of the points with the highest derivative values to be highlighted on the plots.
    ruta_figuras : str
        File path where the generated figure will be saved.
    Returns:
    --------
    None
        The function saves the generated plot as an image file at the specified location.
    Notes:
    ------
    - The function creates a figure with two subplots:
        1. The I-V curve with logarithmic scaling on the y-axis.
        2. The derivative of the I-V curve.
    - Points with the highest derivative values are highlighted in red on both subplots.
    - The figure is saved as an image file in the specified path with a resolution of 300 dpi.
    """

    # Verificar si se pasó un DataFrame
    df = kwargs.get("df", None)
    
    # Crear una figura con dos subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # === Configuración de estilo LaTeX y plot ===
    setup_paper_plt(plt, latex=True, scaling=2)

    config_ax(ax1)
    config_ax(ax2)

    if df is not None:
        voltage_completo = df["Voltaje"]
        current_completo = abs(df["Corriente"])

        # En el primer subplot, graficar la curva I-V
        ax1.plot(abs(voltage_completo), current_completo, "b-", linewidth=2)
        ax1.set_xlabel(r"|Voltaje| (\si{\volt})")
        ax1.set_ylabel(r"Corriente (\si{\ampere})")
        ax1.set_title("Curva I-V")
        ax1.set_yscale("log")
        ax1.grid(True)
        
    else:
        # En el primer subplot, graficar la curva I-V
        ax1.plot(abs(voltaje), intensidad, "b-", linewidth=2)
        ax1.set_xlabel(r"|Voltaje| (\si{\volt})")
        ax1.set_ylabel(r"Corriente (\si{\ampere})")
        ax1.set_title("Curva I-V")
        ax1.set_yscale("log")
        ax1.grid(True)
        
        
    # Solo marcar los puntos con mayor derivada en el primer subplot
    for idx in top_indices:
        ax1.plot(abs(voltaje[idx]), intensidad[idx], "o", markersize=6, color="red")
        ax1.annotate(
            f"{abs(voltaje[idx]):.2f} V",
            (abs(voltaje[idx]), intensidad[idx]),
            textcoords="offset points",
            xytext=(-35, -5),
            ha="center",
            fontsize=16,
            fontweight="bold",
        )

    # En el segundo subplot, graficar la derivada de la corriente respecto al voltaje
    ax2.plot(abs(voltaje), dI_dV, "-", linewidth=2, color="blue")
    ax2.set_xlabel(r"|Voltaje| (\si{\volt})")
    ax2.set_ylabel(r"$\dv{I}{V}$ (\si{\ampere/\volt})")
    ax2.set_title("Derivada de la curva I-V")
    ax2.grid(True)

    # Solo marcar los puntos con mayor derivada en el segundo subplot
    for idx in top_indices:
        ax2.plot(abs(voltaje[idx]), dI_dV[idx], "o", markersize=6, color="red")
        ax2.annotate(
            f"{abs(voltaje[idx]):.2f} V",
            (abs(voltaje[idx]), dI_dV[idx]),
            textcoords="offset points",
            xytext=(-25, -5),
            ha="center",
            fontsize=16,
            fontweight="bold",
        )

    fig.savefig(str(ruta_figuras) + ".pdf", dpi=300, bbox_inches="tight")  # type: ignore
    plt.close(fig)

    # Crear figura combinada con eje secundario
    fig_combinada, ax_comb_1 = plt.subplots(figsize=(12, 9))

    # === Configuración de la figura combinada ===
    
    setup_paper_plt(plt, latex=True, scaling=2)
    config_ax(ax_comb_1)

    if df is not None:
        voltage_completo = df["Voltaje"]
        current_completo = abs(df["Corriente"])

        # ---- Eje izquierdo (Corriente) ----
        ax_comb_1.semilogy(
            abs(voltage_completo),
            current_completo,
            "o-",
            color="blue",
            label=r"Corriente (\si{\ampere})",
        )
        ax_comb_1.set_xlabel(r"|Voltaje| (\si{\volt})")
        ax_comb_1.set_ylabel(r"Corriente (\si{\ampere})", color="blue")
        ax_comb_1.tick_params(axis="y", labelcolor="blue")
    
    else:
        # ---- Eje izquierdo (Corriente) ----
        ax_comb_1.semilogy(
            abs(voltaje), 
            intensidad, 
            "o-", 
            color="blue", 
            label=r"Corriente (\si{\ampere})"
        )
        ax_comb_1.set_xlabel(r"|Voltaje| (\si{\volt})")
        ax_comb_1.set_ylabel(r"Corriente (\si{\ampere})", color="blue")
        ax_comb_1.tick_params(axis="y", labelcolor="blue")

    # Solo marcar los puntos con mayor derivada en el primer subplot
    for idx in top_indices:
        ax_comb_1.plot(abs(voltaje[idx]), intensidad[idx], "o", markersize=6, color="red")
        ax_comb_1.annotate(
            f"{abs(voltaje[idx]):.2f} V",
            (abs(voltaje[idx]), intensidad[idx]),
            textcoords="offset points",
            xytext=(-35, -5),
            ha="center",
            fontsize=16,
            fontweight="bold",
        )

    # ---- Eje derecho (Derivada) ----
    ax_comb_2 = ax_comb_1.twinx()  # Crear un eje Y secundario
    config_ax(ax2)
    ax_comb_2.plot(
        abs(voltaje), dI_dV, "o-", color="orange", label=r"$\dv{I}{V}$ (\si{\ampere/\volt})"
    )
    ax_comb_2.set_ylabel(r"$\dv{I}{V}$ (\si{\ampere/\volt})", color="orange")
    ax_comb_2.tick_params(axis="y", labelcolor="orange")

    # Solo marcar los puntos con mayor derivada en el segundo subplot
    for idx in top_indices:
        ax_comb_2.plot(abs(voltaje[idx]), dI_dV[idx], "o", markersize=6, color="green")
        ax_comb_2.annotate(
            f"{abs(voltaje[idx]):.2f} V",
            (abs(voltaje[idx]), dI_dV[idx]),
            textcoords="offset points",
            xytext=(-25, -5),
            ha="center",
            fontsize=16,
            fontweight="bold",
        )

    # Leyendas combinadas
    lines1, labels1 = ax_comb_1.get_legend_handles_labels()
    lines2, labels2 = ax_comb_2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
    plt.title("I-V y su Derivada")

    fig_combinada.savefig(
        str(ruta_figuras) + "_combinado.pdf", dpi=300, bbox_inches="tight"
    )  # type: ignore
    plt.close(fig_combinada)

In [4]:
def obtener_V_reset_Derivada(ruta_datos, num_valores_maximos=3):
    """
    Analiza un archivo de datos experimentales para calcular la derivada de la corriente
    con respecto al voltaje, identifica los valores máximos de la derivada y genera una
    gráfica de los datos.
    Args:
        ruta_datos (Path): Ruta al archivo que contiene los datos experimentales.
            El archivo debe existir y contener tres columnas: "Voltaje", "Corriente" y "tiempo".
        num_valores_maximos (int, opcional): Número de valores máximos de la derivada a extraer.
            Por defecto es 5.
    Returns:
        dict: Un diccionario con las siguientes claves:
            - "top_derivadas" (np.ndarray): Los valores máximos de la derivada numérica.
            - "top_voltajes" (np.ndarray): Los voltajes correspondientes a los valores máximos
              de la derivada.
    Raises:
        FileNotFoundError: Si el archivo especificado en `ruta_datos` no existe.
    Notas:
        - La función filtra los datos para considerar solo las primeras 111 filas del archivo.
        - Genera una gráfica que muestra la curva IV y la derivada, y la guarda en una
          subcarpeta "Datos_Experimentales/V_Set" con un nombre basado en el archivo de entrada.
    """
    # Verificar que el archivo existe
    if not ruta_datos.exists():
        raise FileNotFoundError(
            f"El archivo {ruta_datos} no existe. Verificar la ruta."
        )

    # Cargar datos experimentales
    data_set = np.loadtxt(ruta_datos)

    # Crear el DataFrame
    df_set = pd.DataFrame(data_set, columns=["Voltaje", "Corriente", "tiempo"])
    df_set = df_set.iloc[0:140]

    # Combinación de ambos filtros
    # Primero, resetea los índices después del filtrado
    df_set_filtrado = df_set.loc[
        (df_set["Voltaje"] >= -1.4) & (df_set["Voltaje"] <= -0.1)
    ].reset_index(drop=True)

    # Calcular la derivada numérica de la corriente con respecto al voltaje
    dI_dV = np.gradient(
        np.log10(abs(df_set_filtrado["Corriente"])), df_set_filtrado["Voltaje"]
    )

    # Extraer los índices de los valores máximos
    indices_maximos = np.argsort(dI_dV)[-num_valores_maximos:]

    # Extraer los valores máximos y sus voltajes correspondientes
    top_derivadas = dI_dV[indices_maximos]
    top_voltages = df_set_filtrado["Voltaje"].values[indices_maximos]

    V_set = df_set_filtrado["Voltaje"]
    I_set = abs(df_set_filtrado["Corriente"])

    ruta_figuras = Path.cwd() / "Datos_Experimentales" / "V_Reset" / f"{ruta_datos.stem}"

    df_figura = pd.DataFrame(
        data_set, columns=["Voltaje", "Corriente", "tiempo"]
    )
    
    plot_iv_and_derivative(
        V_set, I_set, dI_dV, indices_maximos, ruta_figuras, df=df_figura
    )

    # Retornar resultados en un diccionario
    return {
        "top_derivadas": top_derivadas,
        "top_voltajes": top_voltages,
    }


In [ ]:
for datos in archivos:
    
    # Método 1: máximo de la derivada
    resultados = obtener_V_reset_Derivada(datos, num_valores_maximos=3)

    # Encuentra el índice del voltaje más bajo
    idx_min_voltaje = np.argmin(resultados["top_voltajes"])
    voltaje_min = resultados["top_voltajes"][idx_min_voltaje]
    derivada_min = resultados["top_derivadas"][idx_min_voltaje]

    # Método 2: máximo de la intensidad
    data_set = np.loadtxt(datos)
    df_set = pd.DataFrame(data_set, columns=["Voltaje", "Corriente", "tiempo"])

    # Encuentra el índice del valor máximo de la corriente
    idx_max_corriente = df_set["Corriente"].idxmax()

    # Extrae el voltaje correspondiente a ese índice
    voltaje_max_corriente = df_set.loc[idx_max_corriente, "Voltaje"]

    # Guardo los resultados en un único archivo de texto
    with open(ruta_figuras / "V_reset_experimental.txt", "a") as f:
        f.write(f"{datos.stem}\t{voltaje_min:.4f}\t{voltaje_max_corriente:.4f}\n")


In [6]:
results_path = ruta_figuras / "V_Reset_experimental.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),  # String Unicode de máx 50 caracteres
        ("V_reset_derivada", "f8"),  # Float de 64 bits
    ],
    encoding="utf-8",
    names=["Archivo", "V_reset_derivada_V"],
)

print(resultados_txt)

# Crear un DataFrame para una mejor visualización
df_resultados = pd.DataFrame(resultados_txt)

df_resultados["Numero"] = (
    df_resultados["Archivo"].str.extract(r"Cycle_n_(\d+)", expand=False).astype(int)
)

[('Cycle_n_900', -1.13) ('Cycle_n_901', -1.11) ('Cycle_n_902', -1.14)
 ('Cycle_n_903', -1.13) ('Cycle_n_904', -1.14) ('Cycle_n_905', -1.13)
 ('Cycle_n_906', -1.13) ('Cycle_n_907', -1.11) ('Cycle_n_908', -1.14)
 ('Cycle_n_909', -1.11) ('Cycle_n_910', -1.13) ('Cycle_n_911', -1.13)
 ('Cycle_n_912', -1.12) ('Cycle_n_913', -1.11) ('Cycle_n_914', -1.11)
 ('Cycle_n_915', -1.13) ('Cycle_n_916', -1.12) ('Cycle_n_917', -1.1 )
 ('Cycle_n_918', -1.17) ('Cycle_n_919', -1.13) ('Cycle_n_920', -1.13)
 ('Cycle_n_921', -1.14) ('Cycle_n_922', -1.13) ('Cycle_n_923', -1.14)
 ('Cycle_n_924', -1.13) ('Cycle_n_925', -1.12) ('Cycle_n_926', -1.1 )
 ('Cycle_n_927', -1.14) ('Cycle_n_928', -1.11) ('Cycle_n_929', -1.1 )
 ('Cycle_n_930', -1.09) ('Cycle_n_931', -1.09) ('Cycle_n_932', -1.1 )
 ('Cycle_n_933', -1.11) ('Cycle_n_934', -1.14) ('Cycle_n_935', -1.15)
 ('Cycle_n_936', -1.12) ('Cycle_n_937', -1.12) ('Cycle_n_938', -1.14)
 ('Cycle_n_939', -1.11) ('Cycle_n_940', -1.12) ('Cycle_n_941', -1.13)
 ('Cycle_n_942', -1.

In [7]:
# Obtener V_set de los archivos de simulación
ruta_archivos_simulacion = Path.cwd() / "logs"

# Listar todos los archivos
resultados = []

for archivo in ruta_archivos_simulacion.glob("log_simulacion_*.log"):
    with open(archivo, "r", encoding="utf-8") as f:
        contenido = f.read()

    pattern = r"filamento\s*:?\s*(\d+).*?voltaje\s*:?\s*(-?\d*\.?\d+)"
    matches = re.findall(pattern, contenido, re.IGNORECASE)

    if matches:
        # voltajes en el orden en que aparecen en el texto
        voltajes = [float(v) for _, v in matches]
        # formateo a 4 decimales (como en tu código original)
        voltajes_str = [f"{v:.4f}" for v in voltajes]

        linea = [archivo.name] + voltajes_str
        with open(ruta_figuras / "V_reset_simulacion.txt", "a") as f:
            f.write("\t".join(linea) + "\n")

results_path = ruta_figuras / "V_reset_simulacion.txt"

resultados_txt = np.genfromtxt(
    results_path,
    dtype=[
        ("Archivo", "U50"),         # String Unicode de máx 50 caracteres
        ("V_rotura_1", "f8"),       # Float de 64 bits
        ("V_rotura_2", "f8"),       # Float de 64 bits
        ("V_rotura_3", "f8"),       # Float de 64 bits
        ("V_rotura_4", "f8"),       # Float de 64 bits
    ],
    encoding="utf-8",
    names=["Archivo", "V_rotura_1", "V_rotura_2", "V_rotura_3", "V_rotura_4"] # NO se refiere a que se ha roto el filamento 1 2 3 o 4, sino al orden en que aparecen en el log
)

# Crear un DataFrame para una mejor visualización
df_resultados_sim = pd.DataFrame(resultados_txt)
df_resultados_sim["Numero"] = (
    df_resultados_sim["Archivo"]
    .str.extract(r"log_simulacion_(\d+)", expand=False)
    .astype(int)
)

# Elimino la primera columna del dataframe
df_resultados_sim = df_resultados_sim.drop(columns=["Archivo"])

# Muevo la primera columna al de numero al inicio
cols = df_resultados_sim.columns.tolist()
cols = [cols[-1]] + cols[:-1]
df_resultados_sim = df_resultados_sim[cols]
print(df_resultados_sim)

    Numero  V_rotura_1  V_rotura_2  V_rotura_3  V_rotura_4
0        1     -0.7539     -1.0996     -1.2202     -1.3135
1       10     -1.1486     -1.1575     -1.1589     -1.1742
2       11     -0.7867     -1.1094     -1.2732     -1.2933
3       12     -1.1358     -1.1712     -1.1875     -1.2307
4       13     -1.1061     -1.1311     -1.2034     -1.2191
5       14     -1.0308     -1.0794     -1.1196     -1.2145
6       15     -1.0588     -1.1322     -1.2320     -1.2764
7       16     -1.1528     -1.1844     -1.2118     -1.2174
8       17     -1.1319     -1.1542     -1.2142     -1.3741
9       18     -0.8597     -1.1119     -1.1644     -1.2876
10      19     -1.1081     -1.1185     -1.1544     -1.2088
11       2     -1.1414     -1.1640     -1.2925     -1.2943
12      20     -1.1094     -1.1487     -1.1515     -1.2316
13      21     -0.7056     -1.1684     -1.1949     -1.2984
14      22     -1.0907     -1.1819     -1.2243     -1.3254
15      23     -0.9635     -1.1187     -1.1885     -1.27

In [8]:
fig, ax = plt.subplots(figsize=(16, 9))

# === Configuración de estilo LaTeX y plot ===
setup_paper_plt(plt, latex=True, scaling=2)
config_ax(ax)

ax.plot(
    df_resultados["Numero"],
    df_resultados["V_reset_derivada_V"],
    "o",
    color="blue",
    label="Maximum current derivative determination",
)

ax.plot(
    df_resultados_sim["Numero"] + 899,
    df_resultados_sim["V_rotura_1"],
    "D",
    color="green",
    label="Voltage first CF rupture",
)

ax.plot(
    df_resultados_sim["Numero"] + 899,
    df_resultados_sim["V_rotura_2"],
    "s",
    color="red",
    label="Voltage second CF rupture",
)

ax.plot(
    df_resultados_sim["Numero"] + 899,
    df_resultados_sim["V_rotura_4"],
    "<",
    color="purple",
    label="Voltage all CF rupture",
)


ax.set_xlabel(r"Curve number")
ax.set_ylabel(r"Set voltage (\si{\volt})")
ax.legend(
    labelspacing=0.3,
    handletextpad=0.2,
    handlelength=1.0,
    borderaxespad=0.2,
    loc="best",
)
# ax.set_title("Voltaje de Set experimental vs Simulación")
fig.savefig(
    str(Path.cwd() / "Datos_Experimentales" / "V_Reset")
    + "/V_reset_experimental_vs_simulado.pdf",
    dpi=300,
    bbox_inches="tight",
)  # type: ignore
plt.close(fig)